### Import Dependencies

In [87]:
from sentence_transformers import SentenceTransformer
import torch
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct,SparseVectorParams, Modifier, PayloadSchemaType,Document,Prefetch,FusionQuery
import fastembed
import pandas as pd



In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-mpnet-base-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SURYA ER\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Lets see check the embedding model's vector output

In [21]:
print(model.get_embedding_dimension())

768


In [28]:
client = QdrantClient(host="localhost", port=6333)

### Create a collection in qdrant

In [27]:
client.create_collection(
    "steam-data-collection-hybrid-search",
    vectors_config={
        "dense":VectorParams(size=model.get_embedding_dimension(), distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [30]:
client.create_payload_index(
    collection_name = "steam-data-collection-hybrid-search",
    field_name = "appid",
    field_schema = PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Generate embedded data using Sentence Transformer

In [45]:
df = pd.read_csv("../raw data/steam_data_master.csv")

In [49]:
df_steam_json = df.to_dict(orient="records")

In [59]:
def generate_embeddings(text_list, batch_size = 64):

    if not text_list:
        return []

    embedded_vector = model.encode(
        text_list,
        batch_size = batch_size,
        convert_to_numpy = True,
        normalize_embeddings= True,
        show_progress_bar = True
    )

    return embedded_vector.tolist() 

In [55]:
df.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,detailed_description,about_the_game,short_description,header_image,screenshots,background,movies,website,support_url,support_email
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,...,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,https://steamcdn-a.akamaihd.net/steam/apps/10/...,"[{'id': 0, 'path_thumbnail': 'https://steamcdn...",https://steamcdn-a.akamaihd.net/steam/apps/10/...,NaN,NaN,http://steamcommunity.com/app/10,NaN
1,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,...,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,https://steamcdn-a.akamaihd.net/steam/apps/30/...,"[{'id': 0, 'path_thumbnail': 'https://steamcdn...",https://steamcdn-a.akamaihd.net/steam/apps/30/...,NaN,http://www.dayofdefeat.com/,NaN,NaN
2,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,...,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,https://steamcdn-a.akamaihd.net/steam/apps/50/...,"[{'id': 0, 'path_thumbnail': 'https://steamcdn...",https://steamcdn-a.akamaihd.net/steam/apps/50/...,NaN,NaN,https://help.steampowered.com,NaN
3,70,Half-Life,1998-11-08,1,Valve,Valve,windows;mac;linux,0,Single-player;Multi-player;Online Multi-Player...,Action,...,Named Game of the Year by over 50 publications...,Named Game of the Year by over 50 publications...,Named Game of the Year by over 50 publications...,https://steamcdn-a.akamaihd.net/steam/apps/70/...,"[{'id': 0, 'path_thumbnail': 'https://steamcdn...",https://steamcdn-a.akamaihd.net/steam/apps/70/...,NaN,http://www.half-life.com/,http://steamcommunity.com/app/70,NaN
4,80,Counter-Strike: Condition Zero,2004-03-01,1,Valve,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,...,"With its extensive Tour of Duty campaign, a ne...","With its extensive Tour of Duty campaign, a ne...","With its extensive Tour of Duty campaign, a ne...",https://steamcdn-a.akamaihd.net/steam/apps/80/...,"[{'id': 0, 'path_thumbnail': 'https://steamcdn...",https://steamcdn-a.akamaihd.net/steam/apps/80/...,NaN,NaN,http://steamcommunity.com/app/80,NaN


In [56]:
text_to_embed = [
                 data["name"] + " " +
                 data['detailed_description'] + " " +  
                 data['about_the_game'] + " " +
                 data['short_description'] 
                for data in df_steam_json]

In [57]:
len(text_to_embed)

1000

In [60]:
embedded_text  = generate_embeddings(text_to_embed)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [62]:
len(embedded_text[0])

768

### Upload this data to qdrant

##### create pointstructs for the data

In [75]:
pointstructs = []

i = 0

for embedding, data in zip(embedded_text,text_to_embed): 
    pointstructs.append(PointStruct(
        id=df_steam_json[i]["appid"],
        vector={
            "dense":embedding,
            "sparse":Document(
                text=data,
                model="qdrant/bm25"
            )
        },
        payload = df_steam_json[i]
    ))
    i+=1

In [76]:

print(pointstructs[0])

id=10 vector={'dense': [-0.027654651552438736, 0.0041413879953324795, -0.012073020450770855, -0.0015538424486294389, -0.059591103345155716, 0.040910858660936356, -0.03881436586380005, -0.02167011983692646, -0.01832941174507141, 0.022411802783608437, -0.010294735431671143, -0.006956920493394136, 0.02081805281341076, 0.07972389459609985, -0.00761063490062952, 0.025916507467627525, 0.0026604116428643465, -0.026058271527290344, -0.029311325401067734, 0.015777667984366417, 0.0063857766799628735, -0.018875030800700188, -0.060639504343271255, 0.04066668078303337, 0.004812356550246477, -0.05489807203412056, 0.030284373089671135, -0.003977398853749037, 0.061339329928159714, -0.01627608947455883, 0.06290188431739807, -0.05844886973500252, 0.05486202985048294, -0.057710882276296616, 2.4906089493015315e-06, -0.02278684452176094, -0.041541989892721176, -0.03860484063625336, -0.023930924013257027, -0.02494506537914276, 0.03209943696856499, 0.014525153674185276, 0.005034395027905703, 0.02342550829052

In [79]:
client.upsert(collection_name = "steam-data-collection-hybrid-search", points = pointstructs, wait=True)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

##### Let's test it :D

In [82]:
def get_embeddings(text):
    
    embedded_text = model.encode(inputs=text, normalize_embeddings = True, convert_to_numpy = True)
    return embedded_text.tolist()

In [100]:
def retrieve_data(query, limit = 10):


    embedded_query = get_embeddings(query)
    results = client.query_points(collection_name="steam-data-collection-hybrid-search", limit=limit, prefetch=[

        Prefetch(
            query=embedded_query,
            using="dense",
            limit=limit
        ),
        Prefetch(
            query=Document(text=query,model="qdrant/bm25"),
            using="sparse",
            limit=limit
        )
    ], query = FusionQuery(fusion="rrf"))

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_score = []
    retrieved_genres = []
    retrieved_names = []

    for result in results.points:

        retrieved_context_ids.append(result.payload["appid"])
        retrieved_contexts.append(result.payload["detailed_description"])
        similarity_score.append(result.score)
        retrieved_genres.append(result.payload["genres"])
        retrieved_names.append(result.payload["name"])

    return {
        "retrieved_context_ids":retrieved_context_ids,
        "retrieved_contexts":retrieved_contexts,
        "similarity_scores":similarity_score,
        "retrieved_genres":retrieved_genres,
        "retrieved_names": retrieved_names
    }



In [103]:
retrieve_data("suggest me multiplayer fps games")

{'retrieved_context_ids': [500,
  6080,
  10680,
  3270,
  49520,
  32660,
  17080,
  32770,
  13540,
  55100],
 'retrieved_contexts': ['From Valve (the creators of Counter-Strike, Half-Life and more) comes Left 4 Dead, a co-op action horror game for the PC and Xbox 360 that casts up to four players in an epic struggle for survival against swarming zombie hordes and terrifying mutant monsters.                    <br>\t\t\t\t\tSet in the immediate aftermath of the zombie apocalypse, L4D\'s survival co-op mode lets you blast a path through the infected in four unique “movies,” guiding your survivors across the rooftops of an abandoned metropolis, through rural ghost towns and pitch-black forests in your quest to escape a devastated Ground Zero crawling with infected enemies. Each &quot;movie&quot; is comprised of five large maps, and can be played by one to four human players, with an emphasis on team-based strategy and objectives.                     <br>\t\t\t\t\tNew technology dubbed 